# Strategic Allocation – Power BI Visualization

Replaces gut-feel order cutting with a **rules-based tier allocation model**.

**Scenario:** Beefsteak Tomatoes — Total Demand = 8,000 cases, Total Supply = 6,000 cases (−2,000 case shortage).  
**Rule:** Tier 1 (High Margin) takes 0% cuts. Tier 2 (Standard) takes 20% cuts. Tier 3 (Flexible/Promo) takes 50% cuts.

**Run order:** Cell 1 → Cell 2 → Cell 3

> **Note on Cell 3:** When you run it, you will see a message like:  
> *"To sign in, open https://microsoft.com/devicelogin and enter code XXXXXXX"*  
> **This is not an error.** Open that URL in a browser, enter the code, and sign in with
> a Power BI Pro or Premium-per-user account. The report will render automatically once authenticated.

In [ ]:
# ── Cell 1 · Imports ──────────────────────────────────────────────────────────
import os
import pandas as pd

from powerbiclient import QuickVisualize, get_dataset_config
from powerbiclient.authentication import DeviceCodeLoginAuthentication

In [ ]:
# ── Cell 2 · Build Strategic Allocation DataFrame ─────────────────────────────

# ── 2.1  Scenario constants ───────────────────────────────────────────────────
SKU_NAME      = "Beefsteak Tomatoes (15lb 22ct No1)"
TOTAL_DEMAND  = 8_000   # cases ordered across all customers
TOTAL_SUPPLY  = 6_000   # cases available to ship
SHORTAGE      = TOTAL_DEMAND - TOTAL_SUPPLY   # 2,000 cases must be cut

# Cut rates by tier  (must collectively absorb the full 2,000-case shortage)
# Tier 1 demand = 2,500  → cut 0%   = 0 cases
# Tier 2 demand = 2,500  → cut 20%  = 500 cases
# Tier 3 demand = 3,000  → cut 50%  = 1,500 cases
# Total cuts: 0 + 500 + 1,500 = 2,000  ✓
CUT_RATE = {"Tier 1 - High Margin": 0.00,
            "Tier 2 - Standard":    0.20,
            "Tier 3 - Flexible/Promo": 0.50}

# Unit revenue per case by tier (used to calculate Revenue_Saved)
UNIT_PRICE = {"Tier 1 - High Margin":    24.00,
              "Tier 2 - Standard":        20.00,
              "Tier 3 - Flexible/Promo": 16.00}

# Flat (naïve) cut rate = what every customer would face if no strategy was applied
FLAT_CUT_RATE = SHORTAGE / TOTAL_DEMAND   # 2000 / 8000 = 0.25  (25% uniform cut)

# ── 2.2  Customer order data ──────────────────────────────────────────────────
#
# Three tiers, eleven customers, total Original_Order_Qty = 8,000 cases.
#
raw = [
    # Tier 1 – High Margin / Contractual (total: 2,500 cases)
    # These customers have premium contracts or highest margin contribution.
    # Rule: absorb ZERO cuts regardless of shortage depth.
    ("Metro Ontario Inc",          "Tier 1 - High Margin",     900),
    ("Kroger National",            "Tier 1 - High Margin",    1000),
    ("Walmart Premium Account",    "Tier 1 - High Margin",     600),

    # Tier 2 – Standard (total: 2,500 cases)
    # Regular retail customers with standard SLAs.
    # Rule: moderate 20% cut applied evenly across the tier.
    ("Meijer Inc",                 "Tier 2 - Standard",        700),
    ("Sobeys Inc",                 "Tier 2 - Standard",        600),
    ("Loblaws Regional",           "Tier 2 - Standard",        650),
    ("Safeway West",               "Tier 2 - Standard",        550),

    # Tier 3 – Flexible / Promo (total: 3,000 cases)
    # Spot-market buyers, promo deals, secondary distributors, donation programs.
    # Rule: heavy 50% cut — these customers have flexible or non-binding commitments.
    ("Discount Foods East",        "Tier 3 - Flexible/Promo",  600),
    ("Food Service Distributor A", "Tier 3 - Flexible/Promo",  700),
    ("Promo / Secondary Buyer",    "Tier 3 - Flexible/Promo",  500),
    ("Clearance Account",          "Tier 3 - Flexible/Promo",  600),
    ("Donation / Spot Market",     "Tier 3 - Flexible/Promo",  600),
]

df = pd.DataFrame(raw, columns=["Customer_Name", "Priority_Tier", "Original_Order_Qty"])

# ── 2.3  Apply tier-based cut rules ──────────────────────────────────────────
df["Cut_Rate_Applied"] = df["Priority_Tier"].map(CUT_RATE)
df["Suggested_Cut_Qty"]   = (df["Original_Order_Qty"] * df["Cut_Rate_Applied"]).round().astype(int)
df["Final_Allocated_Qty"] = df["Original_Order_Qty"] - df["Suggested_Cut_Qty"]

# Verify total cuts = 2,000 (assert stops the notebook early if data is wrong)
assert df["Suggested_Cut_Qty"].sum() == SHORTAGE, (
    f"Cut total {df['Suggested_Cut_Qty'].sum()} != shortage {SHORTAGE}. Check tier demands."
)
assert df["Final_Allocated_Qty"].sum() == TOTAL_SUPPLY, (
    f"Allocated total {df['Final_Allocated_Qty'].sum()} != supply {TOTAL_SUPPLY}."
)

# ── 2.4  Calculate Revenue_Saved ─────────────────────────────────────────────
#
# Revenue_Saved = revenue retained vs. the naïve flat-cut scenario.
# Under a flat 25% cut, every customer loses 25% of their order regardless of tier.
# Strategic allocation protects Tier 1 fully and concentrates cuts on Tier 3.
#
# Formula:
#   Flat_Allocated_Qty = Original_Order_Qty * (1 - FLAT_CUT_RATE)
#   Revenue_Saved = (Final_Allocated_Qty - Flat_Allocated_Qty) * Unit_Price
#
# Positive value = revenue retained by protecting this customer above the flat baseline.
# Negative value = revenue foregone by cutting this customer more than the flat baseline.
#
df["Unit_Price"]          = df["Priority_Tier"].map(UNIT_PRICE)
df["Flat_Allocated_Qty"]  = (df["Original_Order_Qty"] * (1 - FLAT_CUT_RATE)).round().astype(int)
df["Revenue_Saved"]       = ((df["Final_Allocated_Qty"] - df["Flat_Allocated_Qty"])
                              * df["Unit_Price"]).round(2)

# ── 2.5  Add context columns ──────────────────────────────────────────────────
df["SKU"]              = SKU_NAME
df["Total_Supply"]     = TOTAL_SUPPLY
df["Total_Demand"]     = TOTAL_DEMAND
df["Shortage_Cases"]   = SHORTAGE
df["Fill_Rate_Pct"]    = (df["Final_Allocated_Qty"] / df["Original_Order_Qty"] * 100).round(1)

# ── 2.6  Select and order final columns ──────────────────────────────────────
df_allocation = df[[
    "SKU",
    "Customer_Name",
    "Priority_Tier",
    "Original_Order_Qty",
    "Suggested_Cut_Qty",
    "Final_Allocated_Qty",
    "Fill_Rate_Pct",
    "Unit_Price",
    "Revenue_Saved",
    "Total_Supply",
    "Total_Demand",
    "Shortage_Cases",
]].copy()

# ── 2.7  Print summary ────────────────────────────────────────────────────────
print("=" * 60)
print(f"SKU          : {SKU_NAME}")
print(f"Total Demand : {TOTAL_DEMAND:,} cases")
print(f"Total Supply : {TOTAL_SUPPLY:,} cases")
print(f"Shortage     : {SHORTAGE:,} cases  (flat cut would be {FLAT_CUT_RATE:.0%} across all)")
print("=" * 60)

summary = (
    df_allocation
    .groupby("Priority_Tier", sort=False)
    .agg(
        Customers      = ("Customer_Name",      "count"),
        Total_Ordered  = ("Original_Order_Qty", "sum"),
        Total_Cut      = ("Suggested_Cut_Qty",  "sum"),
        Total_Alloc    = ("Final_Allocated_Qty","sum"),
        Revenue_Saved  = ("Revenue_Saved",       "sum"),
    )
)
summary["Effective_Cut_%"] = (summary["Total_Cut"] / summary["Total_Ordered"] * 100).round(1)
print(summary.to_string())

print("\nTotal Revenue Saved vs flat-cut baseline : $",
      f"{df_allocation[df_allocation['Revenue_Saved'] > 0]['Revenue_Saved'].sum():,.0f}")
print("=" * 60)
print()
df_allocation

In [ ]:
# ── Cell 3 · Render Interactive Power BI Report ───────────────────────────────
#
# WHAT HAPPENS WHEN YOU RUN THIS CELL:
# ─────────────────────────────────────
# 1. A message will appear:
#      "To sign in, use a web browser to open https://microsoft.com/devicelogin
#       and enter the code XXXXXXXX to authenticate."
#    → Open that URL in a browser, enter the code, and sign in.
#    → This is the expected Microsoft device-code flow — NOT an error.
#
# 2. Once authenticated, QuickVisualize uploads df_allocation as a push dataset
#    to the Power BI service, auto-generates a report, and renders it inline
#    in this cell.
#
# REQUIREMENT:
#    The account used to sign in must have a Power BI Pro or
#    Premium-per-user licence.

# Step 1 – Authenticate (triggers the device-code prompt in the cell output)
auth = DeviceCodeLoginAuthentication()

# Step 2 – Convert the DataFrame to a Power BI dataset config
dataset_create_config = get_dataset_config(df_allocation)

# Step 3 – Create and display the interactive report
quick_visualize = QuickVisualize(dataset_create_config, auth=auth)
quick_visualize